# Análisis de ventas — ERD, limpieza y consultas


Este trabajo presenta un modelo ERD para un sistema de ventas, carga un dataset en formato CSV usando pandas, realiza una limpieza completa y responde dos preguntas mediante consultas con filtros y agregaciones.

## 1. ERD

El caso representa una tienda que registra clientes, pedidos, productos y los productos incluidos en cada pedido.

### Entidades y atributos

- **Cliente**: cliente_id (PK), nombre, ciudad, email
- **Pedido**: pedido_id (PK), cliente_id (FK), fecha_pedido, estado
- **Producto**: producto_id (PK), nombre_producto, categoria, precio_unitario
- **DetallePedido**: detalle_id (PK), pedido_id (FK), producto_id (FK), cantidad

### Relaciones y cardinalidad

- Un **Cliente** puede realizar **0..N Pedidos**; cada Pedido pertenece a **1 Cliente**.
- Un **Pedido** contiene **1..N DetallesPedido**; cada DetallePedido pertenece a **1 Pedido**.
- Un **Producto** puede aparecer en **0..N DetallesPedido**; cada DetallePedido corresponde a **1 Producto**.
- Por medio de DetallePedido, **Pedido** y **Producto** tienen una relación **N:M**.

### Diagrama ERD

```mermaid
erDiagram
    CLIENTE ||--o{ PEDIDO : realiza
    PEDIDO ||--|{ DETALLE_PEDIDO : contiene
    PRODUCTO ||--o{ DETALLE_PEDIDO : aparece_en

    CLIENTE {
        int cliente_id PK
        string nombre
        string ciudad
        string email
    }

    PEDIDO {
        int pedido_id PK
        int cliente_id FK
        date fecha_pedido
        string estado
    }

    PRODUCTO {
        int producto_id PK
        string nombre_producto
        string categoria
        decimal precio_unitario
    }

    DETALLE_PEDIDO {
        int detalle_id PK
        int pedido_id FK
        int producto_id FK
        int cantidad
    }
```

## 2. Dataset y limpieza

Para que el archivo sea completamente autocontenido, el dataset está incluido dentro del notebook como texto CSV y se carga con pandas. El dataset simula ventas de una tienda y contiene intencionalmente nulos, duplicados, tipos incorrectos y formatos inconsistentes para demostrar el proceso de limpieza.

In [ ]:
import pandas as pd
import numpy as np
from io import StringIO

# Dataset CSV embebido: no hace falta adjuntar ningún archivo adicional.
csv_data = '''pedido_id,cliente_id,cliente_nombre,ciudad,email,fecha_pedido,producto,categoria,precio_unitario,cantidad,estado
1001,1,Ana Perez,neiva,ana.perez@email.com,2026/01/05,Laptop,Electronica,3200,1,completado
1002,2,Carlos Gomez,NEIVA,carlos.gomez@email.com,05-01-2026,Mouse,electronica,80,3,Completado
1003,3,Maria Lopez,Ibague,maria.lopez@email.com,2026-01-07,Teclado,Oficina,150,2,pendiente
1004,4,Juan Diaz,neiva,,2026-01-08,Monitor,Electronica,900,1,COMPLETADO
1005,5,Sofia Ruiz,BOGOTA,sofia.ruiz@email.com,2026-01-09,Silla,oficina,450,2,completado
1006,6,Pedro Castro,Neiva,pedro.castro@email.com,2026-01-10,Webcam,Electronica,210,2,cancelado
1007,7,Laura Torres,ibague,laura.torres@email.com,2026-01-11,Cuaderno,OFICINA,25,10,completado
1008,8,Diego Mora,Bogota,diego.mora@email.com,2026-01-12,Laptop,electronica,3200,1,completado
1009,9,Valentina Rojas,NEIVA,valentina.rojas@email.com,2026-01-13,Audifonos,Electronica,180,4,pendiente
1010,10,Andres Silva,Neiva,andres.silva@email.com,2026-01-14,Escritorio,Oficina,700,1,completado
1011,11,Camila Leon,ibague,camila.leon@email.com,2026-01-15,Mouse,ELECTRONICA,80,5,completado
1012,12,Martin Gil,Bogota,martin.gil@email.com,2026-01-16,Monitor,Electronica,900,2,COMPLETADO
1013,13,Daniela Cruz,neiva,daniela.cruz@email.com,2026-01-17,Teclado,oficina,150,3,completado
1014,14,Santiago Vega,IBAGUE,santiago.vega@email.com,2026-01-18,Silla,Oficina,450,1,pendiente
1015,15,Paula Nieto,Bogota,paula.nieto@email.com,2026-01-19,Webcam,electronica,210,3,completado
1016,16,Nicolas Rios,neiva,nicolas.rios@email.com,2026-01-20,Audifonos,Electronica,180,2,completado
1017,17,Juliana Marin,NEIVA,juliana.marin@email.com,2026-01-21,Cuaderno,Oficina,25,,completado
1018,18,Felipe Ortiz,bogota,felipe.ortiz@email.com,2026-01-22,Escritorio,oficina,700,2,completado
1019,19,Isabela Soto,Ibague,isabela.soto@email.com,2026-01-23,Laptop,Electronica,3200,1,pendiente
1020,20,Esteban Parra,neiva,esteban.parra@email.com,2026-01-24,Monitor,Electronica,900,1,completado
1021,2,Carlos Gomez,NEIVA,carlos.gomez@email.com,05-01-2026,Mouse,electronica,80,3,Completado
1022,21,Mariana Rojas,NEIVA,mariana.rojas@email.com,2026/01/25,Teclado,Oficina,150,4,completado
1023,22,Samuel Pena,ibague,samuel.pena@email.com,2026-01-26,Silla,OFICINA,450,2,completado
1024,23,Lucia Arias,Bogota,lucia.arias@email.com,2026-01-27,Webcam,Electronica,210,1,completado
1025,24,Mateo Vargas,neiva,mateo.vargas@email.com,2026-01-28,Audifonos,electronica,180,3,COMPLETADO
1026,25,Elena Pardo,NEIVA,elena.pardo@email.com,2026-01-29,Cuaderno,oficina,25,8,completado
'''

df_raw = pd.read_csv(StringIO(csv_data))
print("Dataset cargado correctamente.")
print(f"Filas: {df_raw.shape[0]} | Columnas: {df_raw.shape[1]}")
df_raw.head()

### 2.1 Estado inicial: nulos, duplicados y tipos

In [ ]:
# Resumen inicial
print("Nulos por columna:")
display(df_raw.isna().sum().to_frame("nulos"))

print(f"\nFilas duplicadas exactas: {df_raw.duplicated().sum()}")

print("\nTipos iniciales:")
display(df_raw.dtypes.to_frame("tipo"))

### 2.2 Limpieza

In [ ]:
df = df_raw.copy()

# 1) Normalizar nombres de columnas
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

# 2) Normalizar formatos de texto
text_cols = ["cliente_nombre", "ciudad", "email", "producto", "categoria", "estado"]
for col in text_cols:
    df[col] = df[col].astype("string").str.strip()

df["cliente_nombre"] = df["cliente_nombre"].str.title()
df["ciudad"] = df["ciudad"].str.title()
df["producto"] = df["producto"].str.title()
df["categoria"] = df["categoria"].str.title()
df["estado"] = df["estado"].str.title()

# 3) Corregir tipos
df["pedido_id"] = pd.to_numeric(df["pedido_id"], errors="coerce").astype("Int64")
df["cliente_id"] = pd.to_numeric(df["cliente_id"], errors="coerce").astype("Int64")
df["precio_unitario"] = pd.to_numeric(df["precio_unitario"], errors="coerce")
df["cantidad"] = pd.to_numeric(df["cantidad"], errors="coerce")
df["fecha_pedido"] = pd.to_datetime(df["fecha_pedido"], errors="coerce", dayfirst=False)

# 4) Imputar nulos
# El email faltante se conserva como "no informado".
df["email"] = df["email"].fillna("no informado")
# La cantidad faltante se imputa con la mediana del producto.
df["cantidad"] = df.groupby("producto")["cantidad"].transform(
    lambda s: s.fillna(s.median())
)

# 5) Eliminar duplicados exactos
antes_dup = len(df)
df = df.drop_duplicates().copy()
eliminados_dup = antes_dup - len(df)

# 6) Validación de valores categóricos
categorias_validas = {"Electronica", "Oficina"}
estados_validos = {"Completado", "Pendiente", "Cancelado"}

df = df[df["categoria"].isin(categorias_validas)]
df = df[df["estado"].isin(estados_validos)]

# 7) Variable derivada para las consultas
df["ventas"] = df["precio_unitario"] * df["cantidad"]

print(f"Duplicados eliminados: {eliminados_dup}")
print(f"Filas finales: {len(df)}")
df.head()

### 2.3 Comparación antes/después

In [ ]:
# Comparación compacta
comparacion = pd.DataFrame({
    "Métrica": [
        "Filas",
        "Columnas",
        "Celdas nulas",
        "Filas duplicadas exactas",
        "Tipo fecha",
        "Tipo precio",
        "Tipo cantidad",
        "Formato de ciudad/categoría/estado"
    ],
    "Antes": [
        len(df_raw),
        df_raw.shape[1],
        int(df_raw.isna().sum().sum()),
        int(df_raw.duplicated().sum()),
        str(df_raw["fecha_pedido"].dtype),
        str(df_raw["precio_unitario"].dtype),
        str(df_raw["cantidad"].dtype),
        "Inconsistente (mayúsculas/minúsculas)"
    ],
    "Después": [
        len(df),
        df.shape[1],
        int(df.isna().sum().sum()),
        int(df.duplicated().sum()),
        str(df["fecha_pedido"].dtype),
        str(df["precio_unitario"].dtype),
        str(df["cantidad"].dtype),
        "Normalizado"
    ]
})

display(comparacion)

print("Nulos finales por columna:")
display(df.isna().sum().to_frame("nulos_finales"))

print("Tipos finales:")
display(df.dtypes.to_frame("tipo_final"))

**Resumen de la limpieza:** se detectaron valores nulos, una fila duplicada exacta, fechas almacenadas como texto, valores numéricos que requerían conversión y diferencias de mayúsculas/minúsculas. Los nombres de columnas y campos de texto fueron normalizados. El email faltante se imputó como no informado y la cantidad faltante se imputó usando la mediana de cantidad del mismo producto. Finalmente, se eliminaron los duplicados exactos y se creó la columna ventas = precio_unitario × cantidad.

## 3. Dos preguntas con consultas

### Pregunta 1 — ¿Qué categoría genera más ventas entre los pedidos completados?

In [ ]:
# Filtro + agregación
q1 = (
    df[df["estado"] == "Completado"]
    .groupby("categoria", as_index=False)["ventas"]
    .sum()
    .sort_values("ventas", ascending=False)
)

q1["ventas"] = q1["ventas"].round(2)
display(q1)

ganadora = q1.iloc[0]
print(
    f"Hallazgo: la categoría {ganadora['categoria']} genera el mayor valor de ventas "
    f"entre pedidos completados, con ${ganadora['ventas']:,.2f}."
)

### Pregunta 2 — ¿Qué ciudad tiene el mayor valor promedio por pedido completado?

In [ ]:
# Filtro + agregación
q2 = (
    df[df["estado"] == "Completado"]
    .groupby("ciudad", as_index=False)["ventas"]
    .mean()
    .sort_values("ventas", ascending=False)
)

q2["ventas"] = q2["ventas"].round(2)
display(q2)

mejor_ciudad = q2.iloc[0]
print(
    f"Hallazgo: {mejor_ciudad['ciudad']} tiene el mayor valor promedio por registro "
    f"de venta completada, con ${mejor_ciudad['ventas']:,.2f}."
)

## 4. Conclusión

El análisis demuestra un flujo completo de datos: modelado mediante ERD, carga de un CSV, diagnóstico de calidad, limpieza, normalización, creación de una métrica de ventas y consultas analíticas con filtros y agregaciones. Los resultados permiten identificar tanto la categoría con mayor facturación como la ciudad con mayor valor promedio de venta completada.

## Data & cleaning

This dataset represents sales transactions from a small retail store, including customers, products, categories, order dates, quantities, prices, cities, and order status. The CSV is embedded in this notebook so the assignment can be submitted as a single file without requiring an additional data file. During cleaning, I counted missing values, filled the missing email with “no informado,” and imputed the missing quantity using the median quantity for the same product. I also removed an exact duplicate row, converted dates and numeric fields to appropriate pandas data types, and normalized text formats such as city, category, product, and order status capitalization. Finally, I created a ventas field by multiplying unit price by quantity. The first question asks which product category generates the highest sales value among completed orders, while the second asks which city has the highest average sales value among completed orders. Both questions use pandas filtering and aggregation, and the notebook reports the resulting findings.

Estudiante: Juan David Calderon Gutierrez